# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（交互式 `input()`）
- **输出**：清晰、带例子的解释
- **额外要求**：`stream=True` 流式显示，边生成边用 `update_display` 刷新 Markdown

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「技术专家」答法，user 放具体问题 |
| 流式输出 `stream=True` | 逐 chunk 拼 `response`，刷新同一 `display_id` |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地（OpenAI 兼容） | `base_url='http://localhost:11434/v1'` + `model='llama3.2'` |
| `match / case` | 用数字菜单在两个后端间切换 |

## 怎么跑

1. 从上到下运行单元格；最后一格会进入交互：先选 `1` 或 `2`，再输入问题
2. `.env` 需有可用的 OpenAI 密钥（`load_dotenv`）；选 Llama 时请先 `ollama pull llama3.2` 并保证本机 `11434` 端口可访问
3. 可改 `system_prompt` 的英文指令，对比回答风格（不要翻译成中文再发给模型，除非你有意改行为）


In [ ]:
# ========== 导入：问答工具依赖 ==========

# 标准库 os：供 dotenv / 环境读取使用（本文件主要经 load_dotenv 间接用到）
import os
# load_dotenv：把 .env 读入环境变量，给默认 OpenAI() 提供 OPENAI_API_KEY
from dotenv import load_dotenv
# OpenAI：云端与「Ollama 的 OpenAI 兼容端点」都用同一套 SDK
from openai import OpenAI
# Markdown 展示 + update_display：流式刷新同一块输出区域
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 常量：模型名字集中写在一处 ==========

# OpenAI 云端小模型名（字符串须与账户可用模型一致）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名；需事先 pull，且与 create(model=...) 里用的名字一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境：加载 .env ==========

# override=True：.env 中的键覆盖进程里已有同名变量
load_dotenv(override=True)


In [ ]:
# ========== 系统提示：技术专家角色（英文指令保留，改译会改变答法） ==========

system_prompt = """
        You are a technical expert. You get a question from student, then you answer, clearly, in detail.
        You should explain it with examples.
                """


In [ ]:
# ========== ask_question：交互选题模型 → 流式问答 ==========

def ask_question():
    # 打印菜单（英文选项文案保持原样，与 input 数字对应）
    print("Choose which model you want to ask: \n"
          "1) gpt-4o-mini \n"
          "2) llama3.2 \n")
    # 读用户选择并转成 int：1=云端 GPT，2=本地 Llama
    chosen_model = int(input())

    # 提示输入技术问题
    print("Enter your question: \n")
    # 读一整行用户问题，稍后放进 user message
    user_question = input()

    # stream 先占位；下面 match 分支里赋值为流式迭代器
    stream = None

    # Python 3.10+ 的 match/case：按数字分流到不同后端
    match chosen_model:
        case 1:
            # 默认云端客户端（密钥来自环境变量）
            openai = OpenAI()
            # stream=True：返回可迭代 chunk，而不是一次性完整字符串
            stream = openai.chat.completions.create(model='gpt-4o-mini', messages=[{'role': 'system', 'content': system_prompt},{'role': 'user', 'content': user_question }],
                                           stream=True)
        case 2:
            # 指向本机 Ollama 的 OpenAI 兼容基址；api_key 占位字符串 'ollama' 即可
            openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
            # 模型名 llama3.2 必须与本地已拉取的标签一致
            stream = openai.chat.completions.create(model='llama3.2', messages=[{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_question}],
                                           stream=True)
    # 累积已生成文本，用于每次整段重绘 Markdown
    response = ""
    # 先放一块空 Markdown，拿到 display_id，后续原地更新（避免刷出很多格）
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式 chunk
    for chunk in stream:
        # delta.content 可能为 None（例如角色行）；用 or '' 避免 TypeError
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新为当前完整 Markdown
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 入口：作为脚本/笔记本主格时启动交互 ==========

# 在 notebook 里直接跑本格也会为真；用于触发 ask_question()
if __name__ == '__main__':
    # 进入菜单 + 提问循环（本实现只问一轮）
    ask_question()
